## Inventorized objects

Script to inventorize the objects that the legacy Hive metastore manages or points to

In [3]:
catalog = "legacy_hms_dev"

inventory = []

schemas = [
    r.databaseName
    for r in spark.sql(f"SHOW DATABASES IN {catalog}").collect()
]

for schema in schemas:
    tables = spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect()

    for t in tables:
        table = t.tableName
        full_name = f"{catalog}.{schema}.{table}"

        try:
            desc = spark.sql(f"DESCRIBE EXTENDED {full_name}").collect()

            props = {
                r.col_name.strip(): str(r.data_type).strip()
                for r in desc
                if r.col_name
            }

            inventory.append({
                "schema": schema,
                "table": table,
                "type": props.get("Type"),
                "provider": props.get("Provider"),
                "location": props.get("Location")
            })

        except Exception as e:
            inventory.append({
                "schema": schema,
                "table": table,
                "type": "ERROR",
                "provider": None,
                "location": None
            })

df_inventory = spark.createDataFrame(inventory)

display(
    df_inventory.orderBy("schema", "type", "table")
)

,location,provider,schema,table,type
0,abfss://legacy-hms@stnorthmartdev.dfs.core.windows.net/finance/audit_events,hive,legacy_finance,audit_events,EXTERNAL
1,abfss://legacy-hms@stnorthmartdev.dfs.core.windows.net/finance/payments,hive,legacy_finance,payments,EXTERNAL
2,abfss://legacy-hms@stnorthmartdev.dfs.core.windows.net/finance/regulatory_reporting,hive,legacy_finance,regulatory_reporting,EXTERNAL
3,file:/user/hive/warehouse/legacy_finance.db/accounts,hive,legacy_finance,accounts,MANAGED
4,file:/user/hive/warehouse/legacy_finance.db/budgets,hive,legacy_finance,budgets,MANAGED
5,file:/user/hive/warehouse/legacy_finance.db/cost_centers,hive,legacy_finance,cost_centers,MANAGED
6,file:/user/hive/warehouse/legacy_finance.db/exchange_rates,hive,legacy_finance,exchange_rates,MANAGED
7,file:/user/hive/warehouse/legacy_finance.db/fraud_rules,hive,legacy_finance,fraud_rules,MANAGED
8,file:/user/hive/warehouse/legacy_finance.db/invoices,hive,legacy_finance,invoices,MANAGED
9,file:/user/hive/warehouse/legacy_finance.db/risk_scores,hive,legacy_finance,risk_scores,MANAGED


## Migration strategy

In [2]:
from pyspark.sql.functions import col, when

df_migration = (
    df_inventory
    .withColumn(
        "migration_strategy",
        when(
            (col("type") == "EXTERNAL") &
            col("location").startswith("abfss://"),
            "EXTERNAL_TO_UC"
        )
        .when(
            (col("type") == "MANAGED") &
            col("location").startswith("file:"),
            "COPY_TO_CLOUD_THEN_UC_MANAGED"
        )
        .otherwise("REVIEW")
    )
)

display(df_migration.orderBy("migration_strategy", "schema", "table"))

,location,provider,schema,table,type,migration_strategy
0,file:/user/hive/warehouse/legacy_finance.db/accounts,hive,legacy_finance,accounts,MANAGED,COPY_TO_CLOUD_THEN_UC_MANAGED
1,file:/user/hive/warehouse/legacy_finance.db/budgets,hive,legacy_finance,budgets,MANAGED,COPY_TO_CLOUD_THEN_UC_MANAGED
2,file:/user/hive/warehouse/legacy_finance.db/cost_centers,hive,legacy_finance,cost_centers,MANAGED,COPY_TO_CLOUD_THEN_UC_MANAGED
3,file:/user/hive/warehouse/legacy_finance.db/exchange_rates,hive,legacy_finance,exchange_rates,MANAGED,COPY_TO_CLOUD_THEN_UC_MANAGED
4,file:/user/hive/warehouse/legacy_finance.db/fraud_rules,hive,legacy_finance,fraud_rules,MANAGED,COPY_TO_CLOUD_THEN_UC_MANAGED
5,file:/user/hive/warehouse/legacy_finance.db/invoices,hive,legacy_finance,invoices,MANAGED,COPY_TO_CLOUD_THEN_UC_MANAGED
6,file:/user/hive/warehouse/legacy_finance.db/risk_scores,hive,legacy_finance,risk_scores,MANAGED,COPY_TO_CLOUD_THEN_UC_MANAGED
7,file:/user/hive/warehouse/legacy_finance.db/transactions,hive,legacy_finance,transactions,MANAGED,COPY_TO_CLOUD_THEN_UC_MANAGED
8,file:/user/hive/warehouse/legacy_iot.db/asset_components,hive,legacy_iot,asset_components,MANAGED,COPY_TO_CLOUD_THEN_UC_MANAGED
9,file:/user/hive/warehouse/legacy_iot.db/asset_failures,hive,legacy_iot,asset_failures,MANAGED,COPY_TO_CLOUD_THEN_UC_MANAGED
